In [1]:
!pip install -q imbalanced-learn xgboost

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, roc_auc_score

In [2]:
np.random.seed(42)
n = 1000

data = {
    'amount': np.random.uniform(1, 500, n),
    'distance_from_home': np.random.uniform(0, 100, n),
    'is_online': np.random.choice([0, 1], n),
    'is_fraud': np.random.choice([0, 1], n, p=[0.95, 0.05])
}

df = pd.DataFrame(data)

X = df[['amount', 'distance_from_home', 'is_online']]
y = df['is_fraud']

print(y.value_counts())
df.head()

is_fraud
0    947
1     53
Name: count, dtype: int64


,amount,distance_from_home,is_online,is_fraud
0,187.895519,18.513293,1,0
1,475.406439,54.190095,0,0
2,366.264977,87.294584,0,0
3,299.730584,73.222489,0,1
4,78.853302,80.656115,1,0


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(pd.Series(y_train_resampled).value_counts())

is_fraud
1    758
0    758
Name: count, dtype: int64


In [4]:
svm_model = SVC(probability=True, random_state=42)
svm_model.fit(X_train_resampled, y_train_resampled)
svm_preds = svm_model.predict(X_test_scaled)

In [5]:
xgb_model = XGBClassifier(eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_resampled, y_train_resampled)
xgb_preds = xgb_model.predict(X_test_scaled)

In [6]:
print("--- Baseline SVM Performance ---")
print("ROC-AUC:", roc_auc_score(y_test, svm_model.predict_proba(X_test_scaled)[:, 1]))
print(classification_report(y_test, svm_preds, zero_division=0))

print("\n--- XGBoost Performance ---")
print("ROC-AUC:", roc_auc_score(y_test, xgb_model.predict_proba(X_test_scaled)[:, 1]))
print(classification_report(y_test, xgb_preds, zero_division=0))

--- Baseline SVM Performance ---
ROC-AUC: 0.47089947089947093
              precision    recall  f1-score   support

           0       0.94      0.72      0.81       189
           1       0.04      0.18      0.06        11

    accuracy                           0.69       200
   macro avg       0.49      0.45      0.44       200
weighted avg       0.89      0.69      0.77       200


--- XGBoost Performance ---
ROC-AUC: 0.5067340067340067
              precision    recall  f1-score   support

           0       0.95      0.70      0.81       189
           1       0.07      0.36      0.11        11

    accuracy                           0.69       200
   macro avg       0.51      0.53      0.46       200
weighted avg       0.90      0.69      0.77       200



In [7]:
importances = pd.Series(xgb_model.feature_importances_, index=X.columns)
print(importances)

amount                0.287566
distance_from_home    0.327074
is_online             0.385360
dtype: float32
